# 08주 · 수치적분·분할 조건·낙하 거리

수치해석및실습 CE273

**목표**: 수치 알고리즘·오차·수렴·안정성 이해와 방법 선택의 근거 탐구

학습 주제: 수치적분 (1) / 수치적분 (2)·중간평가 해설

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE273-numerical-analysis/week08/CE273_W08.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE273-numerical-analysis/week08)

모형·가정 확인 → 손계산·첫 갱신 → 단계별 계산 → 스텝·초기값 변경 → 오차·수렴·안정성 설명

셀 실행: Shift+Enter. 처음에는 위에서 아래로 실행. 공통 준비 셀 실행 후 각 절의 개별 실행도 가능. None 항목과 변경 셀은 직접 작성

핵심 실습: 손계산·원리와 계산식의 연결·조건 변경·해석. AI 활용은 개념 학습 후 선택 활동

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE273-numerical-analysis/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)


## 0. 실행 환경·보조 함수

필요한 버전만 설치. 업데이트한 패키지를 이미 불러온 경우 런타임 재시작 후 처음부터 실행

보조 함수의 세부 문법 암기 불필요. 입력·처리·반환값·오차 기준·실패 조건부터 확인. 아래 실습의 호출 코드와 계산식 대조


In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.11", "matplotlib": "3.8", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, linalg, interpolate, integrate

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 자료·최소 원소 수 확인 필요")
    return x

def relative_error(truth, estimate):
    # 기준값 0에서 상대오차를 정의하지 않음. 절대오차 별도 확인
    return np.nan if truth == 0 else abs((truth-estimate)/truth)
def parachute_velocity(t, g=9.81, m=68.1, c=12.5):
    if not (np.isfinite([g,m,c]).all() and g>0 and m>0 and c>=0):
        raise ValueError("g·m 양수, c 비음수 확인 필요")
    t = np.asarray(t, dtype=float)
    if not np.isfinite(t).all() or np.any(t<0):
        raise ValueError("유한한 비음수 시간 필요")
    return g*t if c == 0 else g*m/c*(-np.expm1(-c*t/m))

def parachute_distance(t, g=9.8, m=68.1, c=12.5):
    parachute_velocity(t,g,m,c) # 같은 물리 입력 검증
    t=np.asarray(t,dtype=float)
    if c == 0:
        return .5*g*t*t
    return g*m/c*(t + m/c*np.expm1(-c*t/m))
def quadrature(f, a, b, n, method="trapezoid"):
    if not np.isfinite([a,b]).all() or b<=a or not isinstance(n,int) or not 1<=n<=100000:
        raise ValueError("a<b, 양의 정수 분할 수 확인 필요")
    if method=="simpson13" and n%2:raise ValueError("Simpson 1/3: 짝수 분할 수 필요")
    if method=="simpson38" and n%3:raise ValueError("Simpson 3/8: 3의 배수 분할 수 필요")
    x=np.linspace(a,b,n+1); y=np.asarray(f(x),dtype=float);h=(b-a)/n
    if y.shape!=x.shape or not np.isfinite(y).all():raise ValueError("각 격자점의 유한한 함수값 필요")
    if method=="trapezoid":return h*(.5*y[0]+y[1:-1].sum()+.5*y[-1])
    if method=="simpson13":return h/3*(y[0]+y[-1]+4*y[1:-1:2].sum()+2*y[2:-1:2].sum())
    if method=="simpson38":
        w=np.full(n+1,3.);w[0]=w[-1]=1;w[3:-1:3]=2
        return 3*h/8*np.dot(w,y)
    raise ValueError("적분 방법 확인 필요")


<a id="s01"></a>

## 5차 다항식의 Newton–Cotes

**먼저 예측**: 구간 수 n과 자료점 수 n+1을 구분했는가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록


In [ ]:
f=lambda x:.2+25*x-200*x**2+675*x**3-900*x**4+400*x**5
F=lambda x:.2*x+12.5*x**2-200*x**3/3+675*x**4/4-180*x**5+400*x**6/6
truth=F(.8)-F(0)
rows=[]
for method,n in [("trapezoid",2),("trapezoid",4),("simpson13",4),("simpson38",3)]:
    q=quadrature(f,0,.8,n,method);rows.append((method,n,q,abs(truth-q)))
print(pd.DataFrame(rows,columns=["method","n","estimate","absolute_error"]))
assert np.isclose(truth,1.64053333333333)
assert np.isclose(quadrature(f,0,.8,4,"simpson13"),integrate.simpson(f(np.linspace(0,.8,5)),x=np.linspace(0,.8,5)))

**조건 변경**: Simpson 1/3에 홀수 n을 주고 거절 이유 확인

예제 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**선택 심화 · AI 검토**: 적분구간·등간격·분할 수·방법·해석 적분값·오차 단위 명시

**결과 검토**: 다항식 차수·분할 조건·가중치 합을 검토

예상·중간 계산·변경 비교·독립 검산·원리 설명 기록


In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


<a id="s02"></a>

## 낙하 거리와 g 설정

**먼저 예측**: 속도를 적분한 결과의 단위는?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록


In [ ]:
g,m,c=9.8,68.1,12.5
truth=parachute_distance(10,g,m,c)
for n in [5,10,20,40]:
    estimate=quadrature(lambda t:parachute_velocity(t,g,m,c),0,10,n)
    print(n,estimate,"absolute_error_m:",abs(truth-estimate))
print("원본 해석 거리(m):",truth,"g9.81 변경 시:",parachute_distance(10,9.81,m,c))
assert np.isclose(truth,289.43514651129396)

**조건 변경**: 1주차 g9.81과 8주차 g9.8의 차이를 같은 조건으로 분리 확인

예제 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**선택 심화 · AI 검토**: g9.8m/s²·m68.1kg·c12.5kg/s·t0~10s·분할 수·거리m 명시

**결과 검토**: 원본 차시별 g 차이를 수치적분 오차로 오해하지 않음

예상·중간 계산·변경 비교·독립 검산·원리 설명 기록


In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## 주차 확인

구간·분할 조건·정확해·단위와 입력 차이 검토

입력 항목:

- `다항식_참적분`: 해당 계산값·항 수·오차 기준의 수치
- `Simpson13_홀수분할_허용`: 판단 · Python `True`/`False`
- `낙하거리_g9_8_m`: 해당 계산값·항 수·오차 기준의 수치

공개 계산·개념 확인용 항목. 값만 복사하면 학습 완료로 판단 불가. 가정·적용 범위·검산 근거는 아래 서술과 함께 확인


In [ ]:
answers = {'다항식_참적분': None, 'Simpson13_홀수분할_허용': None, '낙하거리_g9_8_m': None}


In [ ]:
def check_answers(values):
    expected = {'다항식_참적분': 1.640533333333333, 'Simpson13_홀수분할_허용': False, '낙하거리_g9_8_m': 289.43514651129396}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## 원리 탐구·결과 해석 기록

1. 문제·자료·단위·모형과 가정
2. 실행 전 예측·작은 손계산과 근거
3. 기준 계산의 중간값·결과와 원리의 연결
4. 한 조건 변경 결과와 차이의 원인
5. 독립 검산·적용 조건·불확실성·한계
6. 자신의 말로 설명할 핵심 원리 한 가지

AI 활용은 선택. 사용 시 요청·의심한 출력·수정·검산 근거를 별도 기록. 글자 수 자동 채점 없음


In [ ]:
review_log = {
    "문제·자료·단위·가정": None,
    "예측·손계산 근거": None,
    "중간 계산·원리 연결": None,
    "조건 변경·차이의 원인": None,
    "검산·해석·적용 한계": None,
    "내 설명": None,
}
# 선택 활동: AI 활용 시 요청·의심한 출력·수정·검산 근거를 별도 기록
